In [1]:
import genesis as gs
gs.init(logging_level="warning", backend=gs.constants.backend.metal)

[I 12/29/25 20:33:58.598 228158856] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


In [2]:
import torch
from envs.Quadruped import QuadrupedEnv
from rsl_rl.runners.on_policy_runner import OnPolicyRunner
import os


In [ ]:
def get_train_cfg(exp_name, max_iterations):

    train_cfg_dict = {
        "algorithm": {
            "class_name": "PPO",
            "clip_param": 0.2,
            "desired_kl": 0.01,
            "entropy_coef": 0.01,
            "gamma": 0.99,
            "lam": 0.95,
            "learning_rate": 0.0005,
            "max_grad_norm": 1.0,
            "num_learning_epochs": 5,
            "num_mini_batches": 4,
            "schedule": "adaptive",
            "use_clipped_value_loss": True,
            "value_loss_coef": 1.0,
        },
        "init_member_classes": {},
        "policy": {
            "class_name": "ActorCritic",
            "activation": "elu",
            "actor_hidden_dims": [512, 256, 256, 128],
            "critic_hidden_dims": [512, 256, 256, 128],
            "init_noise_std": 1.0,
        },
        "runner": {
            "class_name": "PPO",
            "checkpoint": -1,
            "load_run": -1,
            "log_interval": 1,
            "policy_class_name": "ActorCritic",
            "record_interval": -1,
        },
        "class_name": "OnPolicyRunner",
        "experiment_name": exp_name,
        "run_name": "quadruped_motion",
        "save_interval": 100,
        "num_steps_per_env": 24,
        "seed": 1,
        "logger": "tensorboard",
        "max_iterations": max_iterations,
        "obs_groups": {"policy": ["policy"], "critic": ["policy"]},
    }

    return train_cfg_dict


def get_cfgs():
    env_cfg = {
        "num_actions": 12,
        # joint/link names
        "dof_names": [
            "fl_hx",
            "fr_hx",
            "hl_hx",
            "hr_hx",
            "fl_hy",
            "fr_hy",
            "hl_hy",
            "hr_hy",
            "fl_kn",
            "fr_kn",
            "hl_kn",
            "hr_kn",
        ],
        "default_joint_angles": {  # [rad]
            "fl_hx": 0.0,
            "fr_hx": 0.0,
            "hl_hx": 0.0,
            "hr_hx": 0.0,
            "fl_hy": 0.7,
            "fr_hy": 0.7,
            "hl_hy": 0.7,
            "hr_hy": 0.7,
            "fl_kn": -1.3,
            "fr_kn": -1.3,
            "hl_kn": -1.3,
            "hr_kn": -1.3,
        },
        # PD
        "kp": 350,
        "kd": 50,
        "force_lower": -100,
        "force_upper": 100,
        # termination
        "termination_if_roll_greater_than": 10,  # degree
        "termination_if_pitch_greater_than": 13,
        # base pose
        "base_init_pos": [0.0, 0.0, 0.5546],
        "base_init_quat": [1.0, 0.0, 0.0, 0.0],
        "episode_length_s": 20.0,
        "resampling_time_s": 4.0,
        "action_scale": 0.25,
        "simulate_action_latency": True,
        "clip_actions": 100.0,
    }
    obs_cfg = {
        "num_obs": 46,
        "obs_scales": {
            "lin_vel": 2.0,
            "ang_vel": 0.25,
            "dof_pos": 1.0,
            "dof_vel": 0.05,
        },
    }
    reward_cfg = {
        "tracking_sigma": 0.25,
        "base_height_target": 0.55,
        "crouch_height": 0.25,
        "reward_scales": {
            "tracking_lin_vel": 1.0,
            "tracking_ang_vel": 1.0,
            "lin_vel_z": -0.05,
            "base_height": -50.0,
            "action_rate": -0.005,
            "similar_to_default": -0.4,
            "orientation": -0.002,
        },
    }
    command_cfg = {
        "num_commands": 4,
        "lin_vel_x_range": [-1.0, 2.0],
        "lin_vel_y_range": [-0.5, 0.5],
        "ang_vel_range": [-0.6, 0.6],
        "height_range": [0.25,0.75]
    }

    return env_cfg, obs_cfg, reward_cfg, command_cfg



In [4]:
exp_name = "quadruped_walking_finetuning"

log_dir = f"logs/{exp_name}"
ckpt_start = 12000

## Teach robot to stand still when there are no velocity inputs

In [5]:
def custom_command_sampler(env, envs_idx):
    commands = torch.zeros((len(envs_idx), env.command_cfg["num_commands"]), device=env.device)
    commands[:, 0] = 0.0  # lin_vel_x
    commands[:, 1] = 0.0  # lin_vel_y
    commands[:, 2] = 0.0  # ang_vel
    commands[:, 3] = 0.55  # crouch height
    return commands

In [6]:
env_cfg, obs_cfg, reward_cfg, command_cfg = get_cfgs()
reward_cfg["reward_scales"]["tracking_lin_vel"] = 5

env = QuadrupedEnv(
    num_envs=4096,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    device="mps",
    show_viewer=True,
    custom_command_sampler=custom_command_sampler,
)

iterations = 500
train_cfg = get_train_cfg(exp_name, iterations)

runner = OnPolicyRunner(env, train_cfg, log_dir, device="mps")
resume_path = os.path.join(log_dir, f"model_{ckpt_start}.pt")
runner.load(resume_path)

runner.learn(num_learning_iterations=iterations, init_at_random_ep_len=False)

[Genesis] [20:34:05] [WARNING] Interactive viewer running in main thread. It will only be responsive if a simulation is running.
[Genesis] [20:34:06] [WARNING] Constraint solver time constant should be greater than 2*substep_dt. timeconst is changed from `0.004` to `0.04`). Decrease simulation timestep or increase timeconst to avoid altering the original value.
[Genesis] [20:34:06] [WARNING] Constraint solver time constant should be greater than 2*substep_dt. timeconst is changed from `0.02` to `0.04`). Decrease simulation timestep or increase timeconst to avoid altering the original value.
[Genesis] [20:34:06] [WARNING] Reference robot position exceeds joint limits.
[Genesis] [20:34:07] [WARNING] max_collision_pairs 30 is smaller than the theoretical maximal possible pairs 75, it uses less memory but might lead to missing some collision pairs if there are too many collision pairs


UNSUPPORTED (log once): POSSIBLE ISSUE: unit 4 GLD_TEXTURE_INDEX_CUBE_MAP is unloadable and bound to sampler type (Float) - using zero texture because texture unloadable


[Genesis] [20:34:21] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [20:34:21] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [20:34:21] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [20:34:21] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [20:34:21] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [20:34:21] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [20:34:21] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [20:34:21] [WARNING] This property is depreca

GenesisException: Viewer closed.

In [7]:
env.scene.destroy()